In [12]:
from cytools import fetch_polytopes, Polytope, Cone
from cytools.h_polytope import HPolytope
import numpy as np
import scipy as sp
import itertools
from itertools import combinations
import networkx as nx
from math import gcd, lcm
import sys
import warnings
import gurobipy as gp

import numpy as np
from collections import Counter, defaultdict
from itertools import combinations
from math import factorial
import numpy as np
from scipy.linalg import qr
from cytools import Cone


In [57]:
import sys
sys.path.insert(0, '/Users/Djslime07/Documents/GitHub/Dark-Matter-Abundance-Project/axion-code')
from full_lattice import orthogonal_lattice, LLL_red, extended_euclidean
import matrix as mo

In [58]:
from collections import Counter


class Toric_Variety:
    # defines a simplicial toric variety from a toric fan, given by a set of edges, as points in a lattice, and a set of cones, given by array of edge-labels
    # supported dimensions = 3,2,1,0
    # assumes that input cones form valid simplicial toric fan!
    def __init__(self, edges, cones, edge_labels=None,preferred_line_bundles=None, 
                 env=None, ray_heights=None):
        self._edges = np.array(edges)
        if len(self._edges)>0:
            self._dimension = len(self._edges[0])
        else:
            self._dimension = 0
        if len(set(np.array([len(c) for c in cones])))!=1:
            raise Exception('Only simplicial toric varieties supported.')
        self._cones = cones
        if type(edge_labels)==type(None):
            self._edge_labels = np.arange(len(edges))
        else:
            self._edge_labels = np.array(edge_labels)
            
        self._heights=None
        if type(ray_heights) != type(None):
            self._heights = ray_heights.copy()
            
        self._intersection_numbers = None
        self._is_smooth = None
        self._is_compact = None
        self._torus_invariant_divisors = [None]*len(self._edges)
        self._mori_generators = None
        self._all_simplices = None
        self._labels_to_edge_pos_dict = None
        self._edges_to_labels_dict = None
        self._chi = None
        self._env = env
        self._pt_in_cone = None
        self._l1_tip = None
        self._l2_tip = None
        self._f_secs = None
        self._g_secs = None
        self._alg = None
        self._dual_coxeter = None
        self._decay_constants = None
        self._independent_indices = None
        self._axion_mass_params = None
        self._axion_decay_params = None
        self._axion_masses = None
        if type(preferred_line_bundles)==type(None):
            preferred_line_bundles = np.array([[]]*self._dimension).T
        self._preferred_line_bundles = np.array(preferred_line_bundles)
        
    def __repr__(self):
        if self._dimension==0:
            return f"A point"
        if self._dimension==1:
            return f"A rational curve"
        if self._dimension==2:
            return f"A toric surface"
        if self._dimension==3:
            return f"A toric threefold"
    
    def labels_to_edges(self,label_array):
        return self.edges()[np.array([self.labels_to_edge_positions_dictionary().get(l) for l in label_array])]
    
    def labels_to_edge_positions_dictionary(self):
        if type(self._labels_to_edge_pos_dict)==type(None):
            labels = self.edge_labels()
            self._labels_to_edge_pos_dict = dict([(labels[l],l) for l in range(len(labels))])
        return self._labels_to_edge_pos_dict
        
    
    def edges_to_labels(self,edge_array):
        if type(self._edges_to_labels_dict)==type(None):
            labels = self.edge_labels()
            edges = self.edges()
            self._edges_to_labels_dict = dict([(tuple(edges[i]),labels[i]) for i in range(len(labels))])
        return np.array([self._edges_to_labels_dict.get(tuple(e)) for e in edge_array])
    
    def edges(self):
        return np.array(self._edges)
    
    def cones(self):
        return self._cones
    
    def edge_labels(self):
        return self._edge_labels
    
    def dimension(self):
        return self._dimension
    
    def independent_indices(self, optional_tip=None):
        if type(self._independent_indices) == type(None):
            #do something quick and dirty
            deps = self.linearly_dependent_divisors(sort_type="vol/dual_cox", use_longdoubles=True, optional_tip=optional_tip)
            self._independent_indices = np.array([x for x in range(len(self.edges())) if x not in deps])
        return np.array(self._independent_indices)
    
    def h11(self):
        return len(self.edges()) - 3 #because this is a 3D toric variety
    
    def is_compact(self): #note this replaces the old is_compact function.
        """
        **Description:**
        Determines whether the toric variety is compact by making sure that all top dimensional cones 
        meet exactly (ambient dimension number) of other top dimensional cones at 
        dimension (ambient dimension number - 1) faces.
    
        **Returns:**
        *(boolean)* Whether the toric variety is compact.
        """
        if type(self._is_compact) == type(None):
            cones=self.cones()
            dim = len(self.edges()[0])
            for c in tqdm(cones):
                intersection_lengths = [len(set(c).intersection(cc)) for cc in cones]
                if np.count_nonzero(np.array(intersection_lengths) == dim-1) != dim:
                    self._is_compact=False
                    return self._is_compact
            self._is_compact=True
        
        return self._is_compact
    
    def heights(self, divisor_list = None):
        """
        **Description:**
        Returns the heights of the toric variety's divisors in the tree of blowups.
        If a list of divisors, i.e. ``divisor_list``, is specified the heights of those
        divisors are returned.
        
        **Arguments:**
        ``divisor_list`` *(list, optional)* A list of integers corresponding to the indices
            of divisors
        
        **Returns:**
        *(list)* If ``divisor_list`` is not ``None``, a list of the heights of each divisor provided
        *(dict)* If ``divisor_list`` is ``None``, then a dictionary is returned whose keys are 
            tuples of ray coordinates, and whose values are the heights of those rays in the tree
        """
        if type(self._heights) == type(None):
            return None
        else:
            if type(divisor_list) != type(None):
                return [self._heights[tuple(self._edges[x])] for x in divisor_list]
            else:
                return self._heights.copy()
    
    def all_simplices(self):
        if type(self._all_simplices)==type(None):
            cones = self.cones()
            all_simplices = mo.flatten_top(mo.flatten_top([[[x for x in itertools.combinations(c,i)] for i in range(len(c)+1)] for c in cones]))
            all_simplices = {tuple(s) for s in all_simplices}
            all_simplices = [s for s in all_simplices]
            self._all_simplices = all_simplices
        return self._all_simplices
    
    def singular_cones(self):
        if type(self._singular_cones)==type(None):
            if self.is_smooth():
                self._singular_cones = []
            else:
                all_simplices = self.all_simplices()
                simplices_with_dimension_greater_than_one = [all_simplices[i] for i in np.where([len(s)>1 for s in all_simplices])[0]]
                singular_pos = np.where([not Cone(self.labels_to_edges(np.array(s))).is_smooth() for s in simplices_with_dimension_greater_than_one])[0]
                self._singular_cones = [simplices_with_dimension_greater_than_one[i] for i in singular_pos] 
        return self._singular_cones
    
    def is_smooth(self):
        if type(self._is_smooth)==type(None):
            cones = self.cones()
            self._is_smooth = np.all([abs(abs(np.linalg.det(self.labels_to_edges(c)))-1)<1e-8 for c in cones])
        return self._is_smooth
    
    def intersection_numbers(self, integerize = True, integerize_tol=1e-1):
        if type(self._intersection_numbers)==type(None):
            label_dict = dict([(i,self._edge_labels[i]) for i in range(len(self._edge_labels))])
            inverse_label_dict = dict([(self._edge_labels[i],i) for i in range(len(self._edge_labels))])
            relabeled_cones = np.array([[inverse_label_dict.get(i) for i in c] for c in self.cones()])
            if self.dimension()>0:
                if self.dimension()==3:
                    intnums0 = compute_3fold_intnums(self.edges(),relabeled_cones)
                if self.dimension()==2:
                    intnums0 = compute_2fold_intnums(self.edges(),relabeled_cones)
                if self.dimension()==1:
                    intnums0 = dict([((i,),1) for i in range(2)])
                self._intersection_numbers = dict([(tuple([label_dict.get(i) for i in k]),intnums0.get(k)) for k in intnums0.keys()])
                
                if integerize:
                    intnum_replacement = dict([(tuple([label_dict.get(i) for i in k]),int(round(intnums0.get(k)))) for k in intnums0.keys()])
                    for kee in intnum_replacement.keys():
                        if abs(intnum_replacement[kee] - self._intersection_numbers[kee]) > integerize_tol:
                            print("The bad key is " + str(kee))
                            print("The bad intersection number is " + str(self._intersection_numbers[kee]))
                            raise Exception("The intersection numbers are not integers")
                        if abs(intnum_replacement[kee] - self._intersection_numbers[kee]) > 1e-4:
                            print("The potentially bad key is " + str(kee))
                            print("The potentially bad intersection number is " + str(self._intersection_numbers[kee]))
                    self._intersection_numbers = intnum_replacement.copy()
            else:
                self._intersection_numbers = None
                
        return self._intersection_numbers
    
    def chi(self):
        # computes topological Euler characteristic
        if type(self._chi)==type(None):
            if self.dimension()>0:
                cones = self.cones()
                edges = self.edges()
                top_intnums = np.array([1/abs(np.linalg.det(self.labels_to_edges(c))) for c in cones])
                self._chi = sum(top_intnums)
            else:
                self._chi = 1
        return self._chi
        
    
    def torus_invariant_divisor(self,divisor_label):
        d = divisor_label
        if type(self._torus_invariant_divisors[np.where(self.edge_labels()==d)[0][0]])==type(None):
            labels = self.edge_labels()
            vanishing_pos = np.where(labels==d)[0][0]
            #cones = self.cones()
            cones=np.array(self.cones())
            edges = self.edges()
            vanishing_edge = self.labels_to_edges([d])[0]
            trafo = extended_euclidean(vanishing_edge)[2]
            reduced_cones = cones[np.where([d in c for c in cones])[0]]
            reduced_cones = np.array([[i for i in set(c)-set([d])] for c in reduced_cones])
            reduced_edge_labels = np.sort(np.unique(np.ndarray.flatten(reduced_cones)))
            positions_in_ambient_edge_labels = np.array([np.where(l==labels)[0][0] for l in reduced_edge_labels])
            reduced_edges = np.delete(trafo@(self.labels_to_edges(reduced_edge_labels).T),0,0).T
            rescalings = np.array([gcd(*list(e)) for e in reduced_edges])
            reduced_edges = np.rint(reduced_edges.T/rescalings).astype(int).T
            # collect recorded line bundles and convert to format usable for toric divisor
            pref_line_bundles_old = np.array(self.preferred_line_bundles())
            vanishing_weights = np.array(pref_line_bundles_old).T[vanishing_pos]
            #print("edges " + str(edges))
            #print("trafo.t " + str(trafo.T))
            #print("the weird thing " + str(np.block([[  -np.array([vanishing_weights])   ],
            #                                                    [np.zeros([self.dimension()-1,len(vanishing_weights)])]  ])))
            #print("first product " + str(edges@trafo.T))
            #print("full product " + str(edges@trafo.T@np.block([[  -np.array([vanishing_weights])   ],
            #                                                    [np.zeros([self.dimension()-1,len(vanishing_weights)])]  ])))
            #print("thing we are adding " + str(pref_line_bundles_old.T))
            #print("sum " + str(edges@trafo.T@np.block([[  -np.array([vanishing_weights])   ],
            #                                                    [np.zeros([self.dimension()-1,len(vanishing_weights)])]  ])+pref_line_bundles_old.T))
            equivalent_representations_of_pref_line_bundles_old = (edges@trafo.T@np.block([[  -np.array([vanishing_weights])   ],
                                                                [np.zeros([self.dimension()-1,len(vanishing_weights)])]  ])+pref_line_bundles_old.T).T
            line_bundles_new = equivalent_representations_of_pref_line_bundles_old.T[positions_in_ambient_edge_labels].T/rescalings
            self._torus_invariant_divisors[np.where(self.edge_labels()==d)[0][0]] = Toric_Variety(reduced_edges,reduced_cones,edge_labels=reduced_edge_labels,
                                                                                                  preferred_line_bundles=line_bundles_new)
        return self._torus_invariant_divisors[np.where(self.edge_labels()==d)[0][0]]
    
    def mori_generators(self, verbose=False, print_it=False, save_as_csr_array=True, filename='moriConesPrintout.py', integerize_intnums=True): 
        """
        **Description:**
        Computes the generators of the Mori cone of this toric variety. 

        **Arguments:**
        - `verbose` *(boolean, optional)*: If set to true, this prints out a count of how many of the total 
            curves have been added to the set of generators.
        - `print_it` *(boolean, optional)*: If set to true, this prints the mori generators to file `filename`
            starting with ``m_gens = `` and follows with the generators in list format 
        - `save_as_csr_array` *(boolean, optional)* The default value is true, in which case the generators are returned 
            as a csr_array. If set to false, the generators are returned as a numpy.array
        - `filename` *(string, optional)* If `print_it' is set to true, this prints the mori generators to `filename`
        - `integerize_intnums` *(boolean, optional)* The default value is true, in which case the intersection numbers are
            rounded to integers after they are computed (with tolerance 1e-1). Note that they should be integers for smooth toric varieties.
            if set to `false` they will not be rounded. This also means that if we do not integerize the intersection numbers,
            the mori generators will be float-valued.
            

        **Returns:**
        *(scipy.csr_array)* If `save_as_csr_array` is set to true, this returns a sparse array such that each row is a generator of the Mori cone
        *(numpy.array)*  If `save_as_csr_array` is set to false, this returns a dense array such that each row is a generator of the Mori cone

        **Example:**
        
        ```
        WRITE SOMETHING HERE?
        ```
        """
        if type(self._mori_generators)==type(None):
            cones = self.cones()
            curve_cones = mo.flatten_top([[np.delete(c,i,0) for i in range(len(c))] for c in cones])#this cuts out one index from each cone
            curve_cones = {tuple(c) for c in curve_cones}
            
            intnums = self.intersection_numbers(integerize=integerize_intnums)

            inums_helper = dict()
            for kee in intnums.keys():
                for ct in range(len(kee)):
                    potential_curves = list(itertools.permutations([kee[x] for x in range(len(kee)) if x != ct]))
                    for p_curve in potential_curves:
                        p_curve_tup = tuple(p_curve)
                        _val = intnums[kee]
                        if abs(_val) > 1e-4:
                            if p_curve_tup not in inums_helper.keys():
                                inums_helper[p_curve_tup] = {tuple((kee[ct], _val))}
                            else:
                                inums_helper[p_curve_tup].add(tuple((kee[ct], _val)))
                    
            for k in inums_helper.keys():
                inums_helper[k] = sorted(list(inums_helper[k]))
    
            curve_charges = set()
            for i,c in enumerate(curve_cones):
                if verbose:
                    print(f"{i}/{len(curve_cones)}",end='\r')
                if len(list(c)) != 2:
                    print("This curve cone does not have length two")
                    sys.exit(0)
                charge = list(inums_helper[c])
                curve_charges.add(tuple(charge))
            if verbose:
                print('f')
            curve_charges = list(curve_charges)
            if print_it:
                fyle1 = open(filename, "w") #prints cones to conesTest.sage
                fyle1.write("m_gens = " + str([c for c in curve_charges]))
                fyle1.close()
            rowData = []
            colData = []
            dataData = []
            for rho in range(len(curve_charges)):
                currentCharge = curve_charges[rho]
                for i in currentCharge:
                    rowData.append(rho)
                    colData.append(i[0])
                    dataData.append(i[1])
            rowData = np.array(rowData)
            colData = np.array(colData)
            dataData = np.array(dataData)
            if save_as_csr_array:
                self._mori_generators = sp.sparse.csr_array((dataData, (rowData, colData))) #this 'should' work, but it won't for unknown reasons
            else:
                self._mori_generators = sp.sparse.csr_array((dataData, (rowData, colData))).toarray()
        return self._mori_generators
    
    def line_bundle_is_ample(self,line_bundle):
        # computes if a line bundle, given as array of coefficients in an expansion in prime toric divisors, is ample
        mori_gens = self.mori_generators()
        if self.dimension()==1:
            return sum(line_bundle)>1e-10
        if self.dimension()==2:
            intnums = self.intersection_numbers()
            labels_to_pos_dict = self.labels_to_edge_positions_dictionary()
            doublets = np.array(list(intnums.keys()))
            ij_doublets = doublets[np.where(doublets.T[0]-doublets.T[1]!=0)[0]]
            ii_doublets = doublets[np.where(doublets.T[0]-doublets.T[1]==0)[0]]
            check_dim1 = min(mori_gens@line_bundle)>1e-10
            check_dim2 = (sum([intnums.get(tuple(k),0)*line_bundle[labels_to_pos_dict.get(k[0])]**2 for k in ii_doublets])
                          +2*sum([intnums.get(tuple(k),0)*line_bundle[labels_to_pos_dict.get(k[0])]*line_bundle[labels_to_pos_dict.get(k[1])] for k in ij_doublets]))>1e-10
            return check_dim1 and check_dim2
        if self.dimension()==3:
            intnums = self.intersection_numbers()
            intnums_tensor = np.zeros([len(self.labels())]*3)
            labels_to_pos_dict = self.labels_to_edge_positions_dictionary()
            for k in intnums.keys():
                intnum = intnums.get(k)
                triplet = np.array([labels_to_pos_dict.get(i) for i in k])
                intnums_tensor[triplet[0]][triplet[1]][triplet[2]] = intnum
                intnums_tensor[triplet[0]][triplet[2]][triplet[1]] = intnum
                intnums_tensor[triplet[1]][triplet[0]][triplet[2]] = intnum
                intnums_tensor[triplet[1]][triplet[2]][triplet[0]] = intnum
                intnums_tensor[triplet[2]][triplet[0]][triplet[1]] = intnum
                intnums_tensor[triplet[2]][triplet[1]][triplet[0]] = intnum
            check_dim1 = min(mori_gens@line_bundle)>1e-10
            check_dim2 = min(line_bundle@intnums_tensor@line_bundle)>1e-10
            check_dim3 = (line_bundle@intnums_tensor@line_bundle)@line_bundle>1e-10
            return check_dim1 and check_dim2 and check_dim3
        
    def line_bundle_is_nef(self,line_bundle):
        # computes if a line bundle, given as array of coefficients in an expansion in prime toric divisors, is nef
        mori_gens = self.mori_generators()
        return min(mori_gens@line_bundle)>-1e-10
    
    def line_bundle_is_non_trivial(self,line_bundle):
        glsm = sp.linalg.null_space(self.edges().T).T
        line_bundle_class = glsm@line_bundle
        return sum(abs(line_bundle_class))>1e-10
    
    def preferred_line_bundles(self):
        # returns a list of line bundles that have been specified when this toric variety was defined
        return self._preferred_line_bundles
    
    def sections(self, divisor_label=None, divisor_coeffs=None):
        """
        **Description:**
        Computes the global sections of a line bundle over torus invariant divisor D = \sum_rho a_rho D_rho,
        where D_rho are the prime toric divisors. 

        **Arguments:**
        - `divisor_coeffs` *(array_like, optional)*: A list of coefficients a_rho yielding the torus
            invariant divisor D. If this is not specified, then divisor_label must be specified.
        - `divisor_label` *(natural number)*: In the case where we want to compute the sections over 
            a prime toric divisor, the divisor_index is the index of that divisor. If this is not specified,
            then divisor_coeffs must be specified.
            
        :::note
        Exactly one of `divisor_coeffs` or `divisor_label` must be specified. Otherwise, an
        exception is raised.
        
        :::

        **Returns:**
        *(numpy.array)* An array consisting of M-lattice points, each of which corresponds 
        to a character in the basis of the set of global sections.

        **Example:**
        
        ```
        %run toric_necessities.ipynb
        tv = Toric_Variety(edges=[(1, 0), (0, 1), (-1, -1)], cones=[[0, 1, 2]])
        index_of_first_divisor = tv.edges_to_labels([(1, 0)])[0]
        print(tv.sections(divisor_label=index_of_first_divisor))
        coeffs = [0 for i in range(len(tv.edges()))]
        coeffs[index_of_first_divisor] = 1
        print(tv.sections(divisor_coeffs=coeffs))
        # [[-1  0]
           [-1  1]
           [ 0  0]]
          [[-1  0]
           [-1  1]
           [ 0  0]]
        ```
        """
        if not ((divisor_coeffs is None) ^ (divisor_label is None)):
            raise ValueError("Exactly one of \"divisor_coeffs\" and \"divisor_label\" "
                            "must be specified.")
            
        divs = np.array(self._edges)
        exp_list = []
        if divisor_label is not None:
            exp_list = np.array([0 for i in range(len(divs))])
            exp_list[divisor_label] = 1
        else:
            exp_list = np.array(divisor_coeffs)
            
        ineqs = np.array([np.append(divs[i], exp_list[i]) for i in range(len(divs))])
        hpoly = HPolytope(ineqs)
        return hpoly.points()
    
    def compute_divisor_volumes(self, optional_tip = None, use_l2_tip = False, use_longdouble=False):
        """
        **Description:**
        Computes the volume of the basis divisors at a location in the Kähler
        cone. The default location is the l1 tip of the SKC.

        The volume of the ith divisor is 0.5*kappa_{ijk} t^j t^k.
        
        **Arguments:**
        - `optional_tip` *(arraylike, optional)* A point in Kähler moduli space to evaluate the volume
        - `use_l2_tip` *(boolean, optional)* If set to true, and no `optional_tip` is provided, this 
        evaluates the divisor volumes of the toric variety at the l2 tip of the SKC
        - `use_longdouble` *(boolean, optional)* Whether to use longdoubles in the computation instead of
        regular doubles. This is only really important when the tip is far enough out in the cone that 
        machine precision becomes an issue.
        
        **Returns:**
        *(np.array)* An array where each entry i corresponds to the volume of the ith divisor
        
        **Example:**
        PROVIDE EXAMPLE
        """
        
        if type(optional_tip) == type(None):
            if use_l2_tip:
                tloc = self.l2_tip()
            else:
                tloc = self.l1_tip()
        else:
            tloc = optional_tip
        
        intnums = self.intersection_numbers()
        if not use_longdouble:
            tau = np.zeros(len(self.edges()), dtype=float)
            for ii in intnums:
                c = Counter(ii)
                for j in c.keys():
                    tau[j] += intnums[ii] * np.prod([tloc[k]**(c[k]-(j==k))/factorial(c[k]-(j==k)) for k in c.keys()])
        else:
            tau = np.zeros(len(self.edges()), dtype=np.longdouble)
            for ii in intnums:
                c = Counter(ii)
                for j in c.keys():
                    tau[j] += np.longdouble(intnums[ii]) * np.prod([np.longdouble(tloc[k])**(c[k]-(j==k))/np.longdouble(factorial(c[k]-(j==k)))for k in c.keys()])
            
        return np.array(tau).astype('float64')
    
    def l1_tip(self, c=1, h_scaling=1, use_quad=False, tip_guess = None):
        """
        **Description:** 
        Computes the tip of the c-stretched Kähler cone with respect to the l1 norm
        
        **Arguments:**
        - `c` *(float, optional)* The stretching of the Kähler cone
        - `h_scaling` *(float, optional)* The scaling of the hyperplanes. If you have issues,
            increasing this can help
        - `use_quad` *(boolean, optional)* Whether to use quad precision when computing the tip. 
            This is recommended at large h11 (like ~70000 or greater)
        - `tip_guess` *(list, optional)* This gives a guess for what the tip should look like. The hope is that this
            reduces the amount of time computing the next tip
            
        **Returns:**
        *(np.array)* A vector of minimal l1 norm such that its dot product with all Mori generators is at least one
            
        **Example:**
        PROVIDE EXAMPLE
        """
        if type(self._l1_tip) == type(None):
            m_gens = self.mori_generators()
            self._l1_tip = tip_of_stretched_cone_gurobi(H=m_gens, c=c, p=1, Hscaling=h_scaling, env=self._env, use_quad=use_quad, tip_guess = tip_guess)
        return self._l1_tip
    
    def find_algebra(self):
        """
        **Description:**
        Computes the gauge groups corresponding to each prime toric
        
        **Returns:**
        *(list)* A list of strings where the ith string gives the gauge group of the ith prime toric
        """
        if type(self._alg) == type(None):
            fsecs = self.compute_f_sections()
            gsecs = self.compute_g_sections()
        
            fexps = []
            for fsec in fsecs:
                fexp = []
                for ray in self.edges():
                    fexp.append(ray@fsec.T + 4)
                fexps.append(fexp)
            
            gexps = []
            for gsec in gsecs:
                gexp = []
                for ray in self.edges():
                    gexp.append(ray@gsec.T + 6)
                gexps.append(gexp)
        
            algs = [0 for i in range(len(fexps[0]))]
            for i in range(len(fexps[0])): # go through all points
                df, dg = minfg(i,fexps,gexps)
                dD = min(3*df,2*dg)
                if df >=1 and dg == 1 and dD == 2: 
                    algs[i]='N'
                elif df ==1 and dg >= 2 and dD == 3: 
                    algs[i]='A1'
                elif df >=2 and dg == 2 and dD == 4:
                    if perfectSquare(gexps, i, 2):
                        algs[i]='A2'
                    else:
                        algs[i]='A1m'
                elif df ==2 and dg >= 3 and dD == 6: 
                    if isD4(fexps, gexps, i):
                        algs[i]='D4'
                    elif isZero(gexps,i,3) and not perfectSquare(fexps,i,2):
                        algs[i]='SO7'
                    else:
                        algs[i]='G2'
                elif df >=2 and dg == 3 and dD == 6: 
                    if isD4(fexps, gexps, i):
                        algs[i]='D4'
                    elif isZero(gexps,i,3) and not perfectSquare(fexps,i,2):
                        algs[i]='SO7'
                    else:
                        algs[i]='G2'
                elif df >=3 and dg == 4 and dD == 8: 
                    if perfectSquare(gexps, i, 4):
                        algs[i]='E6'
                    else:
                        algs[i]='F4'
                elif df ==3 and dg >= 5 and dD == 9: 
                    algs[i]='E7'
                elif df >=4 and dg == 5 and dD == 10: 
                    algs[i]='E8'
            
            self._alg = algs
        return self._alg
    
    def compute_f_sections(self):
        """
        **Description:**
        Computes the sections for f
        
        **Returns:**
        *(list)* A list of integers corresponding to the sections of f
        
        **Example:**
        %run toric_necessities.ipynb
        from sec_test_cones import cones
        from sec_test_rays import rays
        tv = Toric_Variety(edges=rays, cones=cones)
        tv.compute_f_sections()
        #array([[0, 0, 0]])

        """
        if type(self._f_secs) == type(None):
            self._f_secs = self.sections(divisor_coeffs=[4 for i in range(len(self.edges()))])
        return self._f_secs
    
    def compute_g_sections(self):
        """
        **Description:**
        Computes the sections for g
        
        **Returns:**
        *(list)* A list of integers corresponding to the sections of g
        
        **Example:**
        %run toric_necessities.ipynb
        from sec_test_cones import cones
        from sec_test_rays import rays
        tv = Toric_Variety(edges=rays, cones=cones)
        tv.compute_g_sections()
        #array([[ 0,  0,  0],
       [-3, -1, -1],
       [ 0,  0,  1],
       [ 0,  1,  0],
       [ 1,  0,  0],
       [-1,  0,  0]])
        
        """
        if type(self._g_secs) == type(None):
            self._g_secs = self.sections(divisor_coeffs=[6 for i in range(len(self.edges()))])
        return self._g_secs
    
        
    def dual_coxeter_numbers(self):
        """
        **Description:**
        Return the dual coxeter numbers associated to the gauge group living on each torus invariant
        divisor.
        
        **Returns:**
        *(np.array)* An array of dual coxeter numbers, where the ith index is the dual coxeter number 
        of the gauge group living on the ith divisor.
        """
        if type(self._dual_coxeter) == type(None):
            Group_to_Dual_Coxeter = {
                '0': 1,
                'N': 1,
                'A1': 2,
                'A1m': 2,
                'A2': 3,
                'D4': 6,
                'E6': 12,
                'E7': 18,
                'E8': 30, 
                'F4': 9,
                'G2': 4,
                'SO7': 6
                }
            self._dual_coxeter = np.array([Group_to_Dual_Coxeter[str(x)] for x in self.find_algebra()])
        return self._dual_coxeter
    
    def compute_kappa_matrix(self, tloc, use_longdoubles = False):
        """
        **Description:**
        Computes the matrix $\kappa_{ijk}t^k$ at a location ``tloc`` in the Kähler cone. 

        **Arguments:**
        - `tloc` *(array_like)*: A vector specifying a location in the Kähler
            cone.
        - `use_longdoubles` *(boolean, optional)* Whether to use longdoubles in the calculation

        **Returns:**
        *(sp.sparse)* The matrix $\kappa_{ijk}t^k$ at the specified location.

        **Example:**
        cones=[(0, 2, 4), (0, 2, 3), (1, 2, 4), (1, 2, 3), (0, 1, 4), (0, 1, 3)]
        rays = [[0, 0, 1], [0, 1, 0], [1, -1, -1], [-1, 0, 0], [1, 0, 0]]
        test_tv = Toric_Variety(edges=rays, cones=cones)
        test_tv.compute_kappa_matrix(tloc=test_tv.l1_tip()).toarray()
        #array([[ 1.,  1.,  1.,  2.,  1.],
        #[ 1.,  1.,  1.,  2.,  1.],
        #[ 1.,  1.,  1.,  2.,  1.],
        #[ 2.,  2.,  2.,  2.,  0.],
        #[ 1.,  1.,  1.,  0., -1.]])
        
        """
        sum_dict = dict()
        num_edges = len(self.edges())
        intnums = self.intersection_numbers(integerize=True)

        #compute \kappa_{ijk} t^k, storing the (i,j) as the keys of sum_dict, with value \kappa_{ijk}t^k
        for kee in list(intnums.keys()):
            potential_info = list(set(itertools.permutations(kee)))
            for triple in potential_info:
                potential_key = tuple([triple[0], triple[1]])
                val = None
                if use_longdoubles:
                    val = np.longdouble(int(round(intnums[kee]))*np.longdouble(tloc[triple[2]]))
                else:
                    val = int(round(intnums[kee]))*tloc[triple[2]]
                if potential_key in sum_dict:
                    sum_dict[potential_key] += val
                else:
                    sum_dict[potential_key] = val
        
        #convert the previous dictionary to a csr_array
        row_data = []
        col_data = []
        data_data = []

        for kee in list(sum_dict.keys()):
            row_data.append(kee[0])
            col_data.append(kee[1])
            data_data.append(sum_dict[kee])

        return sp.sparse.csr_array((data_data, (row_data, col_data)), shape=(num_edges, num_edges))
    
    def linearly_dependent_divisors(self, sort_type="vol", use_longdoubles=False, optional_tip = None, tol=1e-6):
        """
        **Description:**
        Return three divisor indices which are linearly dependent on the other torus invariant
        divisors that maximize the ``sort_type`` property.
    
        **Arguments:**
        -``sort_type`` *(boolean, optional)* Which sort type to pass to sort_divisor_volumes() to use to 
        determine which linearly dependent divisors to drop.
        -``use_longdoubles`` *(boolean, optional)* Whether to use longdoubles when computing divisor volumes
        -``optional_tip`` *(list, optional)* Where to compute the divisor volumes. If ``None``, values are computed at the l1 tip
        -``tol`` *(double)* If values are less than ``tol`` they are treated as zero
    
        **Returns:**
        *(list)* A list of integers giving indices of divisors which are linearly dependent on other 
        divisors, and so can be safely eliminated to form a basis. 
        """
        tip_for_div_vols = None
        if type(optional_tip) == type(None):
            tip_for_div_vols = self.l1_tip()
        else:
            tip_for_div_vols = np.array(optional_tip)
    
        n_lattice_dim = len(self.edges()[0])
        mgens = self.mori_generators()
        intnums = self.intersection_numbers()
        dual_coxeters = self.dual_coxeter_numbers()
        unsorted_div_vols = self.compute_divisor_volumes(optional_tip=tip_for_div_vols, use_longdouble=use_longdoubles)
    
        linrels = [np.array([x[i] for x in self.edges()]) for i in range(n_lattice_dim)]#3d toric variety
        basis_vecs = np.array(linrels)
        dependent_divisors = []
        reversed_order_to_search, sorted_div_vols, sorted_dcs = sort_divisor_volumes(
            unsorted_div_vols=unsorted_div_vols, 
            dual_coxeters=dual_coxeters, 
            sort_type = sort_type)#reversed because small ones are first here...
    
    
        order_to_search = reversed_order_to_search[::-1]
    
        #print("order to search is " + str(order_to_search))
        num_found = 0
        idx = 0
        while num_found < n_lattice_dim: #3 = dimension of tv.
            vec_idx = 0
            while vec_idx < basis_vecs.shape[0]:
                if abs(basis_vecs[vec_idx, order_to_search[idx]]) > tol:
                    dependent_divisors.append(order_to_search[idx])
                    matr_to_subtract = np.array([basis_vecs[y, order_to_search[idx]]*basis_vecs[vec_idx]/basis_vecs[vec_idx, order_to_search[idx]] for y in range(len(basis_vecs))])
                    basis_vecs = basis_vecs - matr_to_subtract
                
                    num_found = num_found + 1
                    vec_idx = basis_vecs.shape[0]
                else:
                    vec_idx = vec_idx + 1
            idx = idx + 1
    
        return dependent_divisors
    
    def compute_inverse_kahler_metric(self, optional_tip = None, stability_param=True, 
                                      do_sparse=False, sort_type="vol/dual_cox",
                                      verbose=False, divisor_basis=None):
        """
        **Description:**
        Computes the inverse Kähler metric $K^{ab} = 4(\tau^a \tau^b - \mathcal{V} \kappa^{abc} t_c),
        where $t_c$ are the Kähler parameters, $tau^a$ are the divisor volumes, $\kappa^{abc}$ are
        the intersection numbers, and $\mathcal{V}$ is the overall volume. 
        
        **Arguments:**
        `optional_tip` *(list, optional)* Location in the Kähler cone giving Kähler parameters
        `stability_param` *(boolean, optional)* Whether to compute using longdoubles. 
            This is recommended for large h11
        `do_sparse` *(boolean, optional)* Whether to do computations sparsely. If this is true, an 
            sp.sparse.linalg.LinearOperator is returned. 
            Otherwise, a (dense) numpy array is returned
        `divisor_basis` *(list, optional)* A list of indices of linearly independent divisors to be 
            used. Note that this being a basis is not checked, so take care!
        `sort_type` *(string, optional)* Which linearly dependent divisors to drop. Choose among 
            `as_is`, `vol/dual_cox`, `vol`, or `logm_a_no_f`. In `as_is`, divisors whose indices are
            largest are dropped first, in `vol`, divisors with the largest volume are dropped first,
            in `vol/dual_cox`, divisors with the largest volume divided by their dual coxeter number
            are dropped first, and in `logm_a_no_f`, divisors whose associated axions would have the
            smallest mass are dropped first.
        
        **Returns:**
        *(np.array)* If do_sparse is false, then a dense numpy array is returned
        *(sp.sparse.linalg.LinearOperator)* If do_sparse is true, a linear operator is returned
        """
        num_divisors = len(self.edges())
        h11 = num_divisors-3
        compute_all_quantities = (h11 < 10000) or keep_dense #
        if do_sparse:
            compute_all_quantities = False
    
        tloc = None
        if type(optional_tip) == type(None):
            tloc = self.l1_tip()
        else:
            tloc = optional_tip
        
        #compute contractions of the intersection numbers
        if verbose:
            print("Computing quantities depending on contractions of intersection numbers")
        km = self.compute_kappa_matrix(tloc=tloc, use_longdoubles = stability_param)
    
        tau = self.compute_divisor_volumes(optional_tip = tloc, use_longdouble=stability_param)
        if min(tau) < 0 and not stability_param:
            warnings.warn("Divisor volumes calculated without longdoubles are negative. Resetting stability parameter and recomputing divisor volumes.")
            stability_param = True
            tau = self.compute_divisor_volumes(optional_tip = tloc, use_longdouble=stability_param)
    
        if min(tau) < 0:
            print("Minimum divisor volume is " + str(min(tau)))
            raise Exception("Divisor volumes are irrevocably negative.")
    
        total_vol = self.compute_volume(optional_tip=tloc, use_longdouble=stability_param)
    
    
        #Put the kappa matrix and divisor volumes in a divisor basis
        if verbose:
            print("Computing divisor basis and putting relevant quantities in basis")
            
        independent_indices = None
        if type(divisor_basis) == type(None):
            dependent_indices = self.linearly_dependent_divisors(sort_type=sort_type, 
                                                    use_longdoubles=stability_param, 
                                                    optional_tip=tloc)
    
            independent_indices = [x for x in range(num_divisors) if x not in dependent_indices]
        else:
            independent_indices = divisor_basis
            
        self._independent_indices = np.array(independent_indices)
        tau_independent = tau[independent_indices]
        km_independent = ((km[independent_indices].T)[independent_indices]).T
    
        if verbose:
            print("Computing decay constants")
                
        if not do_sparse:
            if h11 > 10000:
                warnings.warn("h11 is large, so constructing all matrices densely may crash the kernel")
            k_inv = 4*(np.outer(tau_independent, tau_independent) 
                       - km_independent.toarray()*total_vol)
        else: #we need to work sparsely
            def mv(v):
                tau_indep_on_v = tau_independent@v.T
                km_indep_on_v = km_independent@v.T
                k_inv_on_v = 4*(tau_independent*tau_indep_on_v - total_vol * km_indep_on_v)
                return k_inv_on_v.astype(np.float64)

            k_inv = sp.sparse.linalg.LinearOperator((h11, h11), matvec=mv)
        
        return k_inv.astype('float64')
    
    def compute_cy_volume(self, optional_tip = None):
        return self.compute_volume(optional_tip = optional_tip)
    
    def compute_volume(self, optional_tip = None, use_l2_tip = False, use_longdouble=False):
        """
        **Description:**
        Computes the volume of the Toric Variety at a location in the Kähler cone. The default location is 
        at the l1 tip of the SKC.

        **Arguments:**
        - `optional_tip` *(arraylike, optional)* A point in Kähler moduli space to evaluate the volume.
        if no tip is specified, the volume is evaluated at the l1 tip.
        - `use_l2_tip` *(boolean, optional)* If set to true, and no `optional_tip` is provided,
        this evaluates the volume of the Toric Variety at the l2 tip of the SKC.
        - `use_longdouble` *(boolean, optional)* Whether to use longdoubles in the computation instead of
        regular doubles. This could only potentially be important if the tip is far enough out in the cone that 
        machine precision becomes an issue. However, it seems (in limited testing on Yinan's big example), that 
        using long doubles does not really matter for the overall volume calculation. 
        
        **Returns:**
        *(float)* The volume of the Toric Variety at the specified location.

        **Example:**
        PROVIDE EXAMPLE
        
        ```
        """
        if type(optional_tip) == type(None):
            if use_l2_tip:
                tloc = self.l2_tip()

def tip_of_stretched_cone_gurobi(H, c=1, p=2, Hscaling=1, env=None, verbosity=0, use_quad=False, tip_guess = None):
    """
    **Description:**
    Find the tip of the stretched cone, using LP.

    **Arguments:**
    - `c`: A real positive number specifying the stretching of the cone
        (i.e. the minimum distance to the defining hyperplanes).
    - `p`: The order of the p-norm used in the tip finding. Currently p=0,1,2 are supported
    - `Hscaling`: How much you scale the hyperplanes. If you are hitting issues with large tips,
        then increasing this number (e.g., to 100) can help
    - `env`: The GurobiPy environment. If you need to solve a large problem. Defined like
        params = {
                    "WLSACCESSID": '[PUT ID HERE]',
                    "WLSSECRET": '[PUT SECRET HERE]',
                    "LICENSEID": [PUT LICENSE HERE]
                }
        env = gp.Env(params=params)
    - `verbose`: Whether to print extra info.

    **Returns:**
    A point in the strict interior of the cone. If no point is found then None
    is returned.
    """
    # initialize the model
    # --------------------
    if env is not None:
        model = gp.Model("tip_finder",env=env)
    else:
        model = gp.Model("tip_finder")

    model.setParam('OutputFlag', verbosity>0)
    model.setParam('FeasibilityTol', 1e-6)
    model.setParam('Method', 0)
    #model.setParam('BarHomogeneous', 1)
    if use_quad:
        model.setParam('Quad', 1)
    #model.setParam('NumericFocus', 3)
    # define variables
    # ----------------
    dim = H.shape[1]
    x = model.addMVar((dim,),
                       lb=-float('inf'), ub=float('inf'),
                       vtype=gp.GRB.CONTINUOUS,
                       name="x")
    nx = model.addVar(name="norm_x")
    
    if type(tip_guess) != type(None):
        x.start = tip_guess
        nx.start = np.linalg.norm(tip_guess, ord=1)
    
    # set objective
    # -------------
    model.setObjective(nx, sense=gp.GRB.MINIMIZE)

    # add constraints
    # ---------------
    model.addMConstr(H*Hscaling, x, '>', np.full(H.shape[0],c),
                     name="stretching")
    if p in (0,1,2):
        model.addGenConstrNorm(resvar=nx, vars=x, which=p)
        #model.addConstr(gp.norm(x, which=p) == nx)
    else:
        raise NotImplemented

    # solve the model, fetch solutions
    # --------------------------------
    model.optimize()
    return Hscaling * x.x


def minfg(i,fsecs,gsecs):
    minf, ming = 10^6, 10^6
    for f in fsecs:
        if f[i] < minf: minf = f[i]
    for g in gsecs:
        if g[i] < ming: ming = g[i]
    return (minf, ming)

def algRank(algs):
    rank = 0
    for a in algs:
        if a == 'D4': rank += 4
        elif a == 'G2': rank += 2
        elif a == 'F4': rank += 4
        elif a == 'E6': rank += 6
        elif a == 'E7': rank += 7
        elif a == 'A1': rank += 1
        elif a == 'A1m': rank += 1
        elif a == 'A2': rank += 2
        elif a == 'SO7': rank += 3
    return rank

def isZero(exps, ind, deg): # just trying to determine if some f_i or g_i is 0
    for e in exps: # go through all the exponents
        if e[ind] == deg: # if of the right degree
            return False
    return True
    
def isD4(fexps, gexps, ind):
    fnum, gnum, fcatch, gcatch = 0, 0, 0, 0
    for f in fexps:
        if f[ind] == 2:
            fnum = fnum + 1
            fcatch = f
    for g in gexps:
        if g[ind] == 3:
            gnum = gnum + 1
            gcatch = g
    if fnum != 1 or gnum != 1: return False
    # #print fcatch, gcatch
    #for f in fcatch: 
    #    if f % 3 != 0: return False
    #for g in gcatch: 
    #    if g % 2 != 0: return False
    for i in range(len(fcatch)):
        #if fcatch[i]/3 != gcatch[i]/2: return False
        if fcatch[i]*3 != gcatch[i]*2: return False
    return True


def perfectSquare(exps, ind, deg):
    num, catch = 0, 0
    for e in exps: # go through all the exponents
        if e[ind] == deg: # if of the right degree
            num = num + 1 # add the number of exps with this deg
            catch = e # catch the monomial
    
    if num == 1: # if not one monomial, not perf square
        for c in catch: # check each degree for this monomial
            if c % 2 != 0: # if not even, not a perfect square
                return False
        return True # if made it here, all even
    return False 

def sort_divisor_volumes(unsorted_div_vols, dual_coxeters, sort_type = "vol/dual_cox"):
    """
    **Description**
    Sort the divisor volumes and associated dual coxeter numbers according to ``sort_type``. ``as_is``
    returns the original order. ``vol`` returns divisors sorted from smallest to largest volume.
    ``vol/dual_cox`` returns divisors sorted from smallest to largest value of 
    divisor volume/dual coxeter number. ``logm_a_no_f`` returns divisors sorted from values
    which make logm_a_no_f biggest to smallest. 
    
    **Arguments:**
    -`unsorted_div_vols` *(list)* A list of divisor volumes
    -`dual_coxeters` *(list)* A list of the dual coxeter numbers associated to each divisor
    -`sort_type` *(str)* Which sort to do
    
    **Returns:**
    *(tuple)* The first element is a numpy array whose ith entry is the original index of the ith divisor
    volume returned. The second element is a numpy array of sorted divisor volumes, and the third
    element is a numpy array of the dual coxeter numbers associated to the sorted divisor volumes. 
    """
    types_of_sorts = {"as_is", "vol/dual_cox", "vol", "logm_a_no_f"}
    if sort_type not in types_of_sorts:
        print("Sort type is " + str(sort_type))
        raise Exception("Sort type unsupported")
    
    udv = np.array(unsorted_div_vols)
    dc = np.array(dual_coxeters)
    
    if sort_type == "as_is":
        sorted_indices = np.array([x for x in range(len(udv))])
    
    elif sort_type == "vol":
        sorted_indices = np.argsort(udv)
    
    elif sort_type == "vol/dual_cox":
        sorted_indices = np.argsort(udv/dc)
        
    elif sort_type == "logm_a_no_f":
        vals = 2*np.pi*udv/dc - np.log(udv) #these are the values which make logm_a_no_f biggest to smallest
        sorted_indices = np.argsort(vals)
        
    return sorted_indices, udv[sorted_indices], dc[sorted_indices]

def compute_3fold_intnums(points,triplets,tol=1e-10):
    
    triplets = np.array([np.sort(x) for x in triplets])
    
    triplets_set = {tuple(x) for x in triplets}
    triplet_intnums = np.array([1/np.rint(abs(np.linalg.det(points[i]))) for i in triplets])
    intnums = dict([(tuple(triplets[i]),triplet_intnums[i]) for i in range(len(triplets))]) #3 cones are indeed just cone volume
    
    divisor_graph = nx.Graph()
    divisor_graph.add_nodes_from(np.arange(len(points)))
    for trip in triplets:
        divisor_graph.add_edges_from([e for e in combinations(trip,2)])
    doublets = np.array(list(divisor_graph.edges()))

    rays_to_neighbors_dict = {}
    for i in range(len(points)):
        rays_to_neighbors_dict[i] = tuple(np.sort(tuple(list(divisor_graph.adj[i]))))

    compact_doublets = []
    noncompact_doublets = []
    edges_to_neighbors_dict = {}
    for i in doublets:
        candidates = set(rays_to_neighbors_dict.get(i[0],set())).intersection(set(rays_to_neighbors_dict.get(i[1],set())))
        candidates = np.array([x for x in candidates])
        pos = np.where([(tuple(np.sort([c]+list(i))) in triplets_set) for c in candidates])[0]
        edges_to_neighbors_dict[tuple(i)] = tuple(candidates[pos])
        if len(pos)==2:
            compact_doublets.append(i)
        if len(pos)==1:
            noncompact_doublets.append(i)
    compact_doublets = np.array(compact_doublets)
    noncompact_doublets = np.array(noncompact_doublets)
    
    noncompact_divisors = np.sort(np.unique(np.ndarray.flatten(noncompact_doublets)))
    if len(noncompact_divisors)>0:
        compact_divisors = np.delete(np.arange(len(points)),noncompact_divisors)
    else:
        compact_divisors = np.arange(len(points))

    for d in compact_doublets:
        edge = tuple(d)
        neighbor_indices = edges_to_neighbors_dict.get(edge)
        relevant_points = points[list(edge)+list(neighbor_indices)]
        other_triple_ints = [intnums.get(tuple(np.sort(list(edge)+[neighbor_indices[0]])),0),intnums.get(tuple(np.sort(list(edge)+[neighbor_indices[1]])),0)]
        glsm_vec = sp.linalg.null_space(np.array(relevant_points).T).T
        linrel = sp.linalg.null_space(glsm_vec).T
        lhside = linrel.T[0:2].T
        rhside = linrel.T[2:4].T@other_triple_ints
        reduced_system = np.random.rand(2,3)@np.append(lhside.T,[rhside],axis=0).T
        solution = -np.linalg.inv(reduced_system.T[0:2].T)@reduced_system.T[2]
        double_triplets = [tuple(np.sort(list(edge)+[edge[0]])),tuple(np.sort(list(edge)+[edge[1]]))]
        intnums[double_triplets[0]] = solution[0]
        intnums[double_triplets[1]] = solution[1]

    for i in compact_divisors:
        neighbor_indices = rays_to_neighbors_dict.get(i)
        relevant_points = points[[i]+list(neighbor_indices)]
        other_ints = [intnums.get(tuple(np.sort([i,i,j])),0) for j in neighbor_indices]
        glsm_vec = sp.linalg.null_space(np.array(relevant_points).T).T
        linrel = sp.linalg.null_space(glsm_vec).T
        lhside = linrel.T[0]
        rhside = linrel.T[1:len(relevant_points)].T@other_ints
        reduced_system = np.random.rand(len(lhside))@np.array([lhside,rhside]).T
        solution = -reduced_system[1]/reduced_system[0]
        intnums[(i,i,i)] = solution
    
    non_trivial_pos = np.where(abs(np.array(list(intnums.values())))>tol)[0]
    vals = np.array(list(intnums.values()))[non_trivial_pos]
    keys = np.array(list(intnums.keys()))[non_trivial_pos]
    
    intnums = dict([(tuple(keys[i]),vals[i]) for i in range(len(keys))])
    
    return intnums

def compute_2fold_intnums(points,doublets):
    """
    
    **Description:**
    Computes the double-intersection numbers of a toric surface defined via the rays in points and the toric fan defined by doublets

    Args:
        points (_type_): _description_
        doublets (_type_): _description_

    Returns:
        _type_: _description_
    """    
    
    divs_set = set(mo.flatten_top([[x for x in itertools.combinations(i,1)] for i in doublets]))
    divs = np.array([x[0] for x in divs_set])

    doublet_intnums = np.array([1/abs(np.linalg.det(points[i])) for i in doublets])

    linear_relations = sp.linalg.null_space(sp.linalg.null_space(points.T).T).T

    divs_compact = divs[np.where([len(np.where([x in set(i) for i in doublets])[0])==2 for x in divs])[0]]
    div_intnums = np.array([y[1]/y[0] 
                            for y in [sp.linalg.null_space(np.array([linear_relations.T[np.array([np.delete(j,np.where(j==i)[0],0)[0] 
                                                                                                  for j in doublets[np.where([any(x) 
                                                        for x in i==doublets])[0]]])].T@doublet_intnums[
                                np.where([any(x) for x in i==doublets])[0]],linear_relations.T[i]]).T,rcond=1e-10).T[0] for i in divs_compact]])

    intnums_dictionary = dict((
        [(tuple(doublets[i]),doublet_intnums[i]) for i in range(len(doublets))]
        +[((divs_compact[i],divs_compact[i]),div_intnums[i]) for i in range(len(divs_compact))]))
    return intnums_dictionary

#Compute log10(ma) without directly np.log10(mass data) to avoid the 0.0 for very small mass values
def logm_a(total_volume, divisor_volume, f, C2):
    logmass_a = 1./2.*(np.log10(32.*(np.pi)**3) - 2*np.log10(total_volume) + np.log10(divisor_volume) - 2*np.log10(f) - ((2.*np.pi*divisor_volume/C2)*np.log10(np.e)))
    return logmass_a


def compute_axion_masses_exact(tv, idiot_check=True, optional_tip=None):
    """
    **Description:**
    Compute the exact log_{10}(axion masses/M_planck) given a toric variety. If h11 is too high, one may need
    to worry about numerical instabilities due to $e^{-2 * \pi * \tau/C_2}$ being really small.
    
    **Arguments:**
    `tv` *(Toric_Variety)* The toric variety serving as a base for an elliptic fibration
        giving the CY4 whose axion masses are being computed
    `idiot_check` *(boolean, optional)* Whether to print the eigenvalues of 
        $L^{-1}@\Lambda4 @ (L^{-1})^T$. This should line up with the idiot check in the
        glimmers implementation
    `optional_tip` *(arraylike, optional)* The Kähler parameters. If not supplied, the
        l1 tip of the SKC is used
    
    **Returns:**
    *(list)* A list of axion masses
    """
    
    k_inv = tv.compute_inverse_kahler_metric(optional_tip = optional_tip).astype(np.float64)
    kahler_metric = np.linalg.inv(k_inv)
    big_ell = np.linalg.cholesky(kahler_metric)
    big_ell_inverse = np.linalg.inv(big_ell)
    big_ell_transpose_inverse = big_ell_inverse.T
    
    div_vols = np.array(tv.compute_divisor_volumes(optional_tip = optional_tip, use_longdouble=True)).astype(np.float64)
    dual_coxeters = np.array(tv.dual_coxeter_numbers())
    
    diagonal_terms = np.array([(div_vols[x]*np.exp(-2*np.pi*div_vols[x]/dual_coxeters[x])).astype(np.float64)
                               for x in range(len(div_vols)) if x in tv.independent_indices()])
    diag_matrix = np.diag(diagonal_terms)
    
    vol = tv.compute_volume(optional_tip=optional_tip, use_longdouble=True)
    relevant_matrix = big_ell_inverse@diag_matrix@big_ell_transpose_inverse
    
    evals = np.linalg.eigvalsh(relevant_matrix)
    
    if idiot_check:
        print("evals are " + str(evals))
    
    evals_prime, evecs_prime = np.linalg.eigh(relevant_matrix)
    
    
    print("Norm of difference of evec*eval - matrix@evec" + str([np.linalg.norm(evals_prime[x]*evecs_prime[x] - relevant_matrix@evecs_prime[x])
           for x in range(len(evecs_prime))]))# seems pretty legit...
    
    return 1/2*np.log10(32*(np.pi**3)/(vol**2)*evals)

<>:390: SyntaxWarning: invalid escape sequence '\s'
<>:661: SyntaxWarning: invalid escape sequence '\k'
<>:778: SyntaxWarning: invalid escape sequence '\m'
<>:1194: SyntaxWarning: invalid escape sequence '\p'
<>:390: SyntaxWarning: invalid escape sequence '\s'
<>:661: SyntaxWarning: invalid escape sequence '\k'
<>:778: SyntaxWarning: invalid escape sequence '\m'
<>:1194: SyntaxWarning: invalid escape sequence '\p'
/var/folders/zv/c3gdzc251n3b6p66ll9_1gw80000gn/T/ipykernel_53665/2079360804.py:390: SyntaxWarning: invalid escape sequence '\s'
  Computes the global sections of a line bundle over torus invariant divisor D = \sum_rho a_rho D_rho,
/var/folders/zv/c3gdzc251n3b6p66ll9_1gw80000gn/T/ipykernel_53665/2079360804.py:661: SyntaxWarning: invalid escape sequence '\k'
  Computes the matrix $\kappa_{ijk}t^k$ at a location ``tloc`` in the Kähler cone.
/var/folders/zv/c3gdzc251n3b6p66ll9_1gw80000gn/T/ipykernel_53665/2079360804.py:778: SyntaxWarning: invalid escape sequence '\m'
  Computes t

In [59]:
import numpy as np
from collections import Counter, defaultdict
from itertools import combinations
from math import factorial
import numpy as np
from scipy.linalg import qr
from cytools import Cone

def compute_axion_masses_exact(tv, idiot_check=True, optional_tip=None):
    """
    **Description:**
    Compute the exact log_{10}(axion masses/M_planck) given a toric variety. If h11 is too high, one may need
    to worry about numerical instabilities due to $e^{-2 * \pi * \tau/C_2}$ being really small.
    
    **Arguments:**
    `tv` *(Toric_Variety)* The toric variety serving as a base for an elliptic fibration
        giving the CY4 whose axion masses are being computed
    `idiot_check` *(boolean, optional)* Whether to print the eigenvalues of 
        $L^{-1}@\Lambda4 @ (L^{-1})^T$. This should line up with the idiot check in the
        glimmers implementation
    `optional_tip` *(arraylike, optional)* The Kähler parameters. If not supplied, the
        l1 tip of the SKC is used
    
    **Returns:**
    *(list)* A list of axion masses
    """
    
    k_inv = tv.compute_inverse_kahler_metric(optional_tip = optional_tip).astype(np.float64)
    kahler_metric = np.linalg.inv(k_inv)
    big_ell = np.linalg.cholesky(kahler_metric)
    big_ell_inverse = np.linalg.inv(big_ell)
    big_ell_transpose_inverse = big_ell_inverse.T
    
    div_vols = np.array(tv.compute_divisor_volumes(optional_tip = optional_tip, use_longdouble=True)).astype(np.float64)
    dual_coxeters = np.array(tv.dual_coxeter_numbers())
    
    diagonal_terms = np.array([(div_vols[x]*np.exp(-2*np.pi*div_vols[x]/dual_coxeters[x])).astype(np.float64)
                               for x in range(len(div_vols)) if x in tv.independent_indices()])
    diag_matrix = np.diag(diagonal_terms)
    
    vol = tv.compute_volume(optional_tip=optional_tip, use_longdouble=True)
    relevant_matrix = big_ell_inverse@diag_matrix@big_ell_transpose_inverse
    
    evals = np.linalg.eigvalsh(relevant_matrix)
    
    if idiot_check:
        print("evals are " + str(evals))
    
    evals_prime, evecs_prime = np.linalg.eigh(relevant_matrix)
    
    
    print("Norm of difference of evec*eval - matrix@evec" + str([np.linalg.norm(evals_prime[x]*evecs_prime[x] - relevant_matrix@evecs_prime[x])
           for x in range(len(evecs_prime))]))# seems pretty legit...
    
    return 1/2*np.log10(32*(np.pi**3)/(vol**2)*evals)

def compute_axion_masses_glimmers(tv, do_glimmers=True, idiot_check=True, optional_tip = None):
    """
    **Description:**
    Compute log_{10}(axion masses/M_planck) using the methods in Glimmers given a toric variety. 
    
    **Arguments:**
    `tv` *(Toric_Variety)* The toric variety serving as a base for an elliptic fibration
        giving the CY4 whose axion masses are being computed
    `do_glimmers` *(boolean, optional)* Whether to use qr decomposition to make qab lower diagonal
        rather than leaving it as (L^{-1})^T which is upper diagonal
    `idiot_check` *(boolean, optional)* Whether to print the eigenvalues of qab.T @ Lambda4 @ qab.
        This should line up with the idiot check in the exact computation.
    `optional_tip` *(arraylike, optional)* The Kähler parameters. If it is not supplied, the l1 tip
        of the SKC is used.
    
    **Returns:**
    *(tuple)* A tuple, the first element is a list of decay constants, the second is a list of axion masses
    """
    #All of this is in the original basis. We need to reshuffle the indexing so that 
    #the largest instanton correction is first.
    
    k_inv = tv.compute_inverse_kahler_metric(optional_tip = optional_tip).astype(np.float64)
    #Note that calling compute_inverse_kahler_metric produces a basis
    
    vol = tv.compute_volume(optional_tip = optional_tip, use_longdouble=True)
    div_vols = np.array(tv.compute_divisor_volumes(optional_tip = optional_tip, use_longdouble=True)).astype(np.float64)
    #dual_coxeters = np.array(tv.dual_coxeter_numbers())
    #print(div_vols*np.exp(-2*np.pi*div_vols/dual_coxeters))
    div_vols = np.array([div_vols[x] for x in range(len(div_vols)) if x in tv.independent_indices()])
    dual_coxeters = np.array(tv.dual_coxeter_numbers())
    dual_coxeters = np.array([dual_coxeters[x] for x in range(len(dual_coxeters)) 
                              if x in tv.independent_indices()])
    #print(div_vols*np.exp(-2*np.pi*div_vols/dual_coxeters))
    
    div_vol_sort_indices = np.argsort(-np.log10(div_vols) + 2*np.pi*div_vols/dual_coxeters*np.log10(np.e))
    #div_vol_sort_indices = np.argsort(-div_vols*np.exp(-2*np.pi*div_vols/dual_coxeters))
    resorted_div_vols = np.array([div_vols[x] for x in div_vol_sort_indices])
    resorted_dual_coxeters = np.array([dual_coxeters[x] for x in div_vol_sort_indices])
    resorted_k_inv = (((k_inv[div_vol_sort_indices]).T)[div_vol_sort_indices]).T
    
    kahler_metric = np.linalg.inv(resorted_k_inv)
    big_ell = np.linalg.cholesky(kahler_metric)
    big_ell_inverse = np.linalg.inv(big_ell)
    big_ell_transpose_inverse = big_ell_inverse.T #this is M
    
    qab = None
    if do_glimmers:
        charges_trans, qab = [A.T for A in np.linalg.qr(big_ell_transpose_inverse.T, 
                                                    mode='reduced')]
    else:
        qab = np.array(big_ell_transpose_inverse)
    if idiot_check:
        diagonal_terms = np.array([(resorted_div_vols[x]*
                                    np.exp(-2*np.pi*resorted_div_vols[x]/resorted_dual_coxeters[x])).astype(np.float64)
                                   for x in range(len(resorted_div_vols))])
        diag_matrix = np.diag(diagonal_terms)
        relevant_matrix = qab.T@diag_matrix@qab
        print("eigenvalues of relevant matrix are " + str(np.linalg.eigvalsh(relevant_matrix)))
    
    
    qs = np.diag(qab)
    
    fs = np.abs(1/(2*np.pi*qs)) #Note that these fs are current_fs/(2*pi)
    
    return fs, 1/2*np.log10(8*np.pi*resorted_div_vols)-np.pi*resorted_div_vols/resorted_dual_coxeters*np.log10(np.e) - np.log10(vol) - np.log10(np.abs(fs)) #what if qab negative? Take absolute value :)

def kahler_diagonal_axion_masses(tv, optional_tip = None):
    """
    **Description:**
    Compute log_{10}(axion masses/M_planck) by taking the decay constants to be the square root of the 
    diagonal elements of the Kähler metric.
    
    **Arguments:**
    `tv` *(Toric_Variety)* The toric variety serving as a base for an elliptic fibration
        giving the CY4 whose axion masses are being computed
    `optional_tip` *(arraylike, optional)* The Kähler parameters. If it is not supplied, the l1 tip
        of the SKC is used.
    
    **Returns:**
    *(tuple)* A tuple, the first element is a list of decay constants, the second is a list of log_10
        axion masses
    """
    k_inv = tv.compute_inverse_kahler_metric(optional_tip = optional_tip).astype(np.float64)
    #Note that calling compute_inverse_kahler_metric produces a basis
    
    vol = tv.compute_volume(optional_tip = optional_tip, use_longdouble=True)
    div_vols = np.array(tv.compute_divisor_volumes(optional_tip = optional_tip, use_longdouble=True)).astype(np.float64)
    div_vols = np.array([div_vols[x] for x in range(len(div_vols)) if x in tv.independent_indices()])
    dual_coxeters = np.array(tv.dual_coxeter_numbers())
    dual_coxeters = np.array([dual_coxeters[x] for x in range(len(dual_coxeters)) 
                              if x in tv.independent_indices()])
    div_vol_sort_indices = np.argsort(-np.log10(div_vols) + 2*np.pi*div_vols/dual_coxeters*np.log10(np.e))
    resorted_div_vols = np.array([div_vols[x] for x in div_vol_sort_indices])
    resorted_dual_coxeters = np.array([dual_coxeters[x] for x in div_vol_sort_indices])
    resorted_k_inv = (((k_inv[div_vol_sort_indices]).T)[div_vol_sort_indices]).T
    
    kahler_metric = np.linalg.inv(resorted_k_inv)
    fs = np.sqrt(np.abs(np.diag(kahler_metric)))
    
    return fs, 1/2*np.log10(32*(np.pi**3)*resorted_div_vols)-np.pi*resorted_div_vols/resorted_dual_coxeters*np.log10(np.e) - np.log10(vol) - np.log10(np.abs(fs)) #what if qab negative? Take absolute value :)

def quick_dirty_scalable_axion_masses(tv, optional_tip = None):
    """
    **Description:**
    Compute log_{10}(axion masses/M_planck) by taking the decay constants to be the 1/(square root of the 
    diagonal elements of the inverse Kähler metric).
    
    **Arguments:**
    `tv` *(Toric_Variety)* The toric variety serving as a base for an elliptic fibration
        giving the CY4 whose axion masses are being computed
    `optional_tip` *(arraylike, optional)* The Kähler parameters. If it is not supplied, the l1 tip
        of the SKC is used.
    
    **Returns:**
    *(tuple)* A tuple, the first element is a list of decay constants, the second is a list of log_10
        axion masses
    """
    k_inv = tv.compute_inverse_kahler_metric(optional_tip = optional_tip).astype(np.float64)
    #Note that calling compute_inverse_kahler_metric produces a basis
    
    vol = tv.compute_volume(optional_tip = optional_tip, use_longdouble=True)
    div_vols = np.array(tv.compute_divisor_volumes(optional_tip = optional_tip, use_longdouble=True)).astype(np.float64)
    div_vols = np.array([div_vols[x] for x in range(len(div_vols)) if x in tv.independent_indices()])
    dual_coxeters = np.array(tv.dual_coxeter_numbers())
    dual_coxeters = np.array([dual_coxeters[x] for x in range(len(dual_coxeters)) 
                              if x in tv.independent_indices()])
    div_vol_sort_indices = np.argsort(-np.log10(div_vols) + 2*np.pi*div_vols/dual_coxeters*np.log10(np.e))
    resorted_div_vols = np.array([div_vols[x] for x in div_vol_sort_indices])
    resorted_dual_coxeters = np.array([dual_coxeters[x] for x in div_vol_sort_indices])
    resorted_k_inv = (((k_inv[div_vol_sort_indices]).T)[div_vol_sort_indices]).T
    
    fs = 1/np.sqrt(np.abs(np.diag(resorted_k_inv)))
    
    return fs, 1/2*np.log10(32*(np.pi**3)*resorted_div_vols)-np.pi*resorted_div_vols/resorted_dual_coxeters*np.log10(np.e) - np.log10(vol) - np.log10(np.abs(fs)) #what if qab negative? Take absolute value :)


###

def compute_axion_masses_current(tv):
    
    newtv = tv
    
    tvplot = []
    h11p = []
    #morigens = []
    tips = []
    mori = []
    kahler = []
    D_list = []
    G_list = []
    idepD_list = []
    idepG_list = []
    f_list = []
    V_list = []

    moriGens = newtv.mori_generators().toarray()  #Get the generators of the Mori cone and time it
    mori_cone = Cone(moriGens)
    mori.append(mori_cone)
    kahler_cone = mori_cone.dual_cone()
    kahler.append(kahler_cone)
    tip = kahler_cone.tip_of_stretched_cone(c=1)
    #morigens.append(moriGens)
    tips.append(tip)
        
    gauge_group = newtv.find_algebra()
    divisor_volume = compute_divisor_volumes(newtv, tip).tolist()
    total_V = compute_cy_volume(newtv, tip)
    # Get inverse kahler metrics and the indices for divisors which is not independent
    K_inv, indices_remove = compute_inverse_kahler_metric(newtv, tip, mori_cone)
        
    # remove the divisor volumes and gauge groups which is not independent
    indep_D = [element for idx, element in enumerate(divisor_volume) if idx not in indices_remove]
    idepD_list.append(indep_D)
    indep_G = [element for idx, element in enumerate(gauge_group) if idx not in indices_remove]
    idepG_list.append(indep_G)
        
    K = np.linalg.inv(K_inv)
    #compute the eigenvalues, get f. Take .real to make sure all are real
    eigenvalues_Kinv = np.linalg.eigvals(K_inv).tolist()
    eigenvalues_K = np.linalg.eigvals(K)
    eigenvalues_K = eigenvalues_K.real
    f = np.sqrt(eigenvalues_K)
    f_decay = f.tolist()
    f_list.append(f_decay)
    copied_f = np.array(f_list)
    
    #set up dict from gauge group to dual coexter
    Group_to_Dual_Coxeter = {
        '0': 1,
        'N': 1,
        'A1': 2,
        'A1m': 2,
        'A2': 3,
        'D4': 6,
        'E6': 12,
        'E7': 18,
        'E8': 30, 
        'F4': 9,
        'G2': 4,
        'SO7': 6
    }
    
    log_mass = []
    effective_mass = []
    mass_G = defaultdict(list)
    divisor_v_for_G = idepD_list
    decay_constant_list = [copied_f[0]]
    total_volume_list = [total_V]
    h11_list = [len(newtv.edges())-3]
    G_list = idepG_list
    
    for group_list, divisor_list, f_list, total_volume, h11 in zip(G_list, divisor_v_for_G, decay_constant_list, total_volume_list, h11_list):
        logmass_list = []
        for name, tau, f in zip(group_list, divisor_list, f_list):
            name_str = str(name)  # Convert to string to ensure consistency with dictionary keys
            assigned_number = Group_to_Dual_Coxeter[name_str]
        
            logm = logm_a(total_volume, tau, f, assigned_number)
            logmass_list.extend([logm])
    
    return copied_f, logmass_list, tips, indices_remove, K
    #return copied_f, logmass_list, tips

###
    
def compute_cy_volume(self, tloc):
        """
        **Description:**
        Computes the volume of the Calabi-Yau at a location in the Kähler cone.

        **Arguments:**
        - `tloc` *(array_like)*: A vector specifying a location in the Kähler
            cone.

        **Returns:**
        *(float)* The volume of the Calabi-Yau at the specified location.

        **Example:**
        We construct a Calabi-Yau hypersurface and find its volume at the tip
        of the stretched Kähler cone.
        ```python {5}
        p = Polytope([[1,0,0,0],[0,1,0,0],[0,0,1,0],[0,0,0,1],[-1,-1,-6,-9]])
        t = p.triangulate()
        cy = t.get_cy()
        tip = cy.toric_kahler_cone().tip_of_stretched_cone(1)
        cy.compute_cy_volume(tip)
        # 3.4999999988856496
        ```
        """
        
        intnums = self.intersection_numbers(integerize = True, integerize_tol=1e-4)
        xvol = 0
        #basis = self.divisor_basis()
        for ii in intnums:
            mult = np.prod([factorial(c)
                            for c in Counter(ii).values()])
            xvol += int(round(intnums[ii]))*np.prod([tloc[int(j)] for j in ii])/mult
        return xvol
    
def compute_divisor_volumes(self, tloc, in_basis=False):
        """
        **Description:**
        Computes the volume of the basis divisors at a location in the Kähler
        cone.

        The volume of the ith divisor is 0.5*kappa_{ijk} t^j t^k.

        **Arguments:**
        - `tloc` *(array_like)*: A vector specifying a location in the Kähler
            cone.
        - `in_basis` *(bool, optional, default=False)*: When set to True, the
            volumes of the current basis of divisors are computed. Otherwise,
            the volumes of all prime toric divisors are computed.

        
        **Example:**
        We construct a Calabi-Yau hypersurface and find the volumes of the
        prime toric divisors at the tip of the stretched Kähler cone.
        ```python {5}
        p = Polytope([[1,0,0,0],[0,1,0,0],[0,0,1,0],[0,0,0,1],[-1,-1,-6,-9]])
        t = p.triangulate()
        cy = t.get_cy()
        tip = cy.toric_kahler_cone().tip_of_stretched_cone(1)
        cy.compute_divisor_volumes(tip)
        # array([ 2.5       , 23.99999999, 16.        ,  2.5       ,  2.5       ,
        #         0.5       ])
        ```
        """
        
        intnums = self.intersection_numbers(integerize = True, integerize_tol=1e-4)
        #basis = self.divisor_basis()
        #if len(basis.shape) == 2: # If basis is matrix
        tau = np.zeros(len(self.edges()), dtype=float)
        for ii in intnums:
            c = Counter(ii)
            for j in c.keys():
                tau[j] += int(round(intnums[ii])) * np.prod([tloc[k]**(c[k]-(j==k))/factorial(c[k]-(j==k)) for k in c.keys()])
        return np.array(tau)
def compute_kappa_matrix(self, tloc):
        """
        **Description:**
        Computes the matrix $\kappa_{ijk}t^k$ at a location in the Kähler cone.

        :::note
        This function only supports Calabi-Yau 3-folds.
        :::

        **Arguments:**
        - `tloc` *(array_like)*: A vector specifying a location in the Kähler
            cone.

        **Returns:**
        *(numpy.ndarray)* The matrix $\kappa_{ijk}t^k$ at the specified
            location.

        **Aliases:**
        `compute_AA`.

        **Example:**
        We construct a Calabi-Yau hypersurface and compute this matrix at the
        tip of the stretched Kähler cone.
        ```python {5}
        p = Polytope([[1,0,0,0],[0,1,0,0],[0,0,1,0],[0,0,0,1],[-1,-1,-6,-9]])
        t = p.triangulate()
        cy = t.get_cy()
        tip = cy.toric_kahler_cone().tip_of_stretched_cone(1)
        cy.compute_kappa_matrix(tip)
        # array([[ 1.,  1.],
        #        [ 1., -3.]])
        ```
        """
        intnums = self.intersection_numbers(integerize = True, integerize_tol=1e-4)
        AA = np.zeros((len(self.edges()),)*2, dtype=float)
        for ii in intnums:
            ii_list = Counter(ii).most_common(3)
            if len(ii_list)==1:
                AA[ii_list[0][0],ii_list[0][0]] += int(round(intnums[ii]))*tloc[ii_list[0][0]]
            elif len(ii_list)==2:
                AA[ii_list[0][0],ii_list[0][0]] += int(round(intnums[ii]))*tloc[ii_list[1][0]]
                AA[ii_list[0][0],ii_list[1][0]] += int(round(intnums[ii]))*tloc[ii_list[0][0]]
                AA[ii_list[1][0],ii_list[0][0]] += int(round(intnums[ii]))*tloc[ii_list[0][0]]
            elif len(ii_list)==3:
                AA[ii_list[0][0],ii_list[1][0]] += int(round(intnums[ii]))*tloc[ii_list[2][0]]
                AA[ii_list[1][0],ii_list[0][0]] += int(round(intnums[ii]))*tloc[ii_list[2][0]]
                AA[ii_list[0][0],ii_list[2][0]] += int(round(intnums[ii]))*tloc[ii_list[1][0]]
                AA[ii_list[2][0],ii_list[0][0]] += int(round(intnums[ii]))*tloc[ii_list[1][0]]
                AA[ii_list[1][0],ii_list[2][0]] += int(round(intnums[ii]))*tloc[ii_list[0][0]]
                AA[ii_list[2][0],ii_list[1][0]] += int(round(intnums[ii]))*tloc[ii_list[0][0]]
            else:
                raise Exception("Error: Inconsistent intersection numbers.")
        return AA
    
def compute_inverse_kahler_metric(self, tloc, mori):
        """
        **Description:**
        Computes the inverse Kähler metric at a location in the Kähler cone.

        :::note
        This function only supports Calabi-Yau 3-folds.
        :::

        **Arguments:**
        - `tloc` *(array_like)*: A vector specifying a location in the Kähler
            cone.

        **Returns:**
        *(numpy.ndarray)* The inverse Kähler metric at the specified location.

        **Example:**
        We construct a Calabi-Yau hypersurface and compute the inverse Kähler
        metric at the tip of the stretched Kähler cone.
        ```python {5}
        p = Polytope([[1,0,0,0],[0,1,0,0],[0,0,1,0],[0,0,0,1],[-1,-1,-6,-9]])
        t = p.triangulate()
        cy = t.get_cy()
        tip = cy.toric_kahler_cone().tip_of_stretched_cone(1)
        cy.compute_inverse_kahler_metric(tip)
        # array([[11., -9.],
        #        [-9., 43.]])
        ```
        """
        xvol = compute_cy_volume(self, tloc)
        #print(xvol)
        Tau = compute_divisor_volumes(self, tloc)
        
        AA = compute_kappa_matrix(self, tloc)
        #print("AA is " + str(AA))
        #AAinv = np.linalg.inv(AA)
        
        Q, R, P = qr(mori.rays(), pivoting=True)#P is a permutation of the rows of mori rays s.t. R is nonincreasing

        # Determine the rank by checking the diagonal of R
        tol = 1e-10
        rank = np.sum(np.abs(np.diag(R)) > tol)

# Identify dependent rows
        dependent_row_indices = np.where(np.abs(np.diag(R)[rank:]) <= tol)[0] + rank

        indices_to_remove = P[dependent_row_indices].tolist()

        indep_Tau = [element for idx, element in enumerate(Tau) if idx not in indices_to_remove]#so we literally drop the divisor volumes of the linearly dependent ones

        mask = np.ones(AA.shape, dtype=bool) #this is boolean...

# Set the rows and columns to remove as False in the mask
        for idx in indices_to_remove: #keep only the parts of AA which are not on the dependent indices
            mask[idx, :] = False
            mask[:, idx] = False

# Apply the mask to the matrix
        indep_AA = AA[mask].reshape(AA.shape[0] - len(indices_to_remove), AA.shape[1] - len(indices_to_remove))
        #indep_AAinv = np.linalg.inv(indep_AA)
        #K = (1./4.)*((np.outer(tloc,tloc)/(2.*(xvol**(2)))) - (indep_AAinv/xvol))
        #print("indep_AA*xvol is " + str(indep_AA*xvol))
        #print("indep_Tau is " + str(indep_Tau))
        Kinv = 4*(np.outer(indep_Tau, indep_Tau) - indep_AA*xvol)
        #print("Kinv is " + str(Kinv))
        #K = np.linalg.inv(Kinv)
        #eigenvalues = np.linalg.eigvals(Kinv)
        return Kinv, indices_to_remove

#Compute log10(ma) without directly np.log10(mass data) to avoid the 0.0 for very small mass values
def logm_a(total_volume, divisor_volume, f, C2):
    logmass_a = 1./2.*(np.log10(32.*(np.pi)**3) - 2*np.log10(total_volume) + np.log10(divisor_volume) - 2*np.log10(f) - ((2.*np.pi*divisor_volume/C2)*np.log10(np.e)))
    return logmass_a

<>:13: SyntaxWarning: invalid escape sequence '\p'
<>:358: SyntaxWarning: invalid escape sequence '\k'
<>:13: SyntaxWarning: invalid escape sequence '\p'
<>:358: SyntaxWarning: invalid escape sequence '\k'
/var/folders/zv/c3gdzc251n3b6p66ll9_1gw80000gn/T/ipykernel_53665/3791594080.py:13: SyntaxWarning: invalid escape sequence '\p'
  to worry about numerical instabilities due to $e^{-2 * \pi * \tau/C_2}$ being really small.
/var/folders/zv/c3gdzc251n3b6p66ll9_1gw80000gn/T/ipykernel_53665/3791594080.py:358: SyntaxWarning: invalid escape sequence '\k'
  Computes the matrix $\kappa_{ijk}t^k$ at a location in the Kähler cone.


In [61]:
from bs4 import BeautifulSoup
import random
import requests
from cytools import Polytope
import numpy as np

def get_polys_rays_and_cones(poly, include_points_interior_to_facets=True):
    """
    ***Description:***
    Triangulates `poly` and returns the rays and cones 
    
    ***Arguments:***
    `poly` *(Polytope)* The polytope to triangulate
    `include_points_interior_to_facets` *(boolean, optional)* Whether to include points interior to facets when triangulating
    
    ***Returns:***
    *(tuple)* A tuple of two lists. The first list's elements are rays, the second's elements are cones
        
    ***Example:***
    p1_poly = Polytope([(1, 0, 0), (0, 1, 0), (0, 0, 1), (-1, -1, -1)])
    get_polys_rays_and_cones(p1_poly)
    #([array([[-1, -1, -1],
         [ 0,  0,  1],
         [ 0,  1,  0],
         [ 1,  0,  0]])],
    #[[(1, 2, 3), (0, 1, 2), (0, 1, 3), (0, 2, 3)]]
    """
    tv = poly.triangulate(include_points_interior_to_facets=include_points_interior_to_facets).get_toric_variety()
    rs = poly.points()[1:]#this gets rid of the 0 ray
    cns = []
    for c in tv.fan_cones():
        cns.append(tuple([np.where(np.all(rs == w, axis=1))[0][0] for w in c.rays()]))
    return rs, cns

def fetch_3d_polytopes(limit=1000, random_polys=True, desired_polys = None):
    """
    ***Description:***
    Return `limit` many 3-D polytopes from the KS database. If indices of polytopes are specified,
    i.e. if `desired_polys` is not None, then it returns those polytopes. Otherwise if `random_polys`
    is set to True, it returns `limit` many random polytopes. Otherwise it returns the first `limit`
    many polytopes.
    
    ***Arguments:***
    `limit` *(int, optional)* How many polytopes to return. If this is not specified, 1000 are returned.
    `random_polys` *(boolean, optional)* Whether to return random polytopes
    `desired_polys` *(list, optional)* If provided, it returns the polytopes whose indices are specified by the list
    
    ***Returns:***
    *(list)* A list of `limit` many polytopes
    """
    if desired_polys is not None:
        polys_to_sample = desired_polys.copy()
    elif limit > 4319:
        raise ValueError("limit cannot exceed 4319, the size of the database")
    elif random_polys:
        polys_to_sample = random.sample(range(4319), limit)
    else:
        polys_to_sample = [x for x in range(limit)]
        
    r = requests.get("http://hep.itp.tuwien.ac.at/%7Ekreuzer/pub/K3/RefPoly.d3")
    soup = BeautifulSoup(r.text, "html.parser")
    lines = soup.text.split("\n")
    
    polytope_list = []
    for num in polys_to_sample:
        first_entry = [int(w) for w in lines[4*num + 1].split(" ") if w != '']
        second_entry = [int(w) for w in lines[4*num + 2].split(" ") if w != '']
        third_entry = [int(w) for w in lines[4*num + 3].split(" ") if w != '']
        polytope_list.append(Polytope(list(zip(first_entry, second_entry, third_entry))))
    
    return polytope_list


In [ ]:
import numpy as np
import pandas as pd
from cytools import fetch_polytopes
import itertools

h11_dict = {
    200:1
}

TAU_MAX = 90.0          # instanton cutoff
LOG_LAMBDA_MIN = -700   # fail safe for instanton scale
Mpl = 1.0

all_axion_data = []

for h11, n_cy3 in h11_dict.items():
    print(f"\n=== Processing h11 = {h11}, {n_cy3} CY3s ===")

    polys = fetch_polytopes(
        h11=h11,
        lattice="N",
        favorable=True,
        limit=n_cy3
    )

    for cy_index, p in enumerate(polys):
        print(f"\n--- CY3 {cy_index} at h11={h11} ---")         

        t = p.triangulate(backend="topcom")                     
        cy = t.get_cy()

        tip = cy.toric_kahler_cone().tip_of_stretched_cone(1)
        vol = cy.compute_cy_volume(tip)

        taus_full = np.array(cy.compute_divisor_volumes(tip))
        K_inv = np.array(cy.compute_inverse_kahler_metric(tip), dtype=np.float64)

        indep = cy.divisor_basis()
        indep_0based = indep - 1 
        taus = taus_full[indep_0based]

        K = np.linalg.inv(K_inv)
        L_inv = np.linalg.cholesky(K_inv)
        L_inv = np.linalg.cholesky(K_inv).T

        evals_check = np.linalg.eigvalsh(K)
        if np.any(evals_check <= 0):
            print(f"  K not positive definite (min eval = {evals_check.min():.3e}), skipping")
            continue
        if not np.all(np.isfinite(K)):
            print(f"  K has non-finite entries, skipping")
            continue

        L = np.linalg.cholesky(K)
        L_inv = np.linalg.inv(L)

        Group_to_Dual_Coxeter = {
        '0': 1, 'N': 1, 'A1': 2, 'A1m': 2, 'A2': 3,
        'D4': 6, 'E6': 12, 'E7': 18, 'E8': 30,
        'F4': 9, 'G2': 4, 'SO7': 6
        }
        tv = cy.ambient_variety()
        triang = p.triangulate(backend="topcom")
        edges = triang.polytope().points()[1:]  
        cones = triang.simplices() 

        tv = Toric_Variety(edges, cones)
        algebra_full = tv.find_algebra()
        print(f"algebra_full: {algebra_full}")
        algebra_indep = [algebra_full[i] for i in indep_0based]
        c2 = np.array([Group_to_Dual_Coxeter[str(g)] for g in algebra_indep])

        try:
            fsecs = tv.compute_f_sections()
            gsecs = tv.compute_g_sections()
            print(f"f_sections shape: {np.array(fsecs).shape}")
            print(f"g_sections shape: {np.array(gsecs).shape}")
        except Exception as e:
            print(f"Section computation failed: {e}")

        active_mask = taus <= TAU_MAX
        diag_terms = np.where(
            active_mask,
            taus * np.exp(-2 * np.pi * taus / c2),
            0.0
        )
        Lambda_diag = np.diag(diag_terms)

        M = L_inv @ Lambda_diag @ L_inv.T
        evals = np.linalg.eigvalsh(M)
        evals = evals[evals > 0]

        if len(evals) == 0:
            print(f"  No positive eigenvalues, skipping")
            continue

        log10_masses = 0.5 * np.log10(32 * np.pi**3 / vol**2 * evals)

        print(f"  vol = {vol:.4f}")
        print(f"  taus = {taus}")
        print(f"  c2   = {c2}")
        print(f"  log10(m/Mpl) = {log10_masses}")